In [1]:
!apt-get install openjdk-17-jdk-headless -qq > /dev/null
!pip install pyspark -q

E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/o/openjdk-17/openjdk-17-jre-headless_17.0.20%2b8-1%7e24.04_amd64.deb  404  Not Found [IP: 91.189.91.82 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/o/openjdk-17/openjdk-17-jdk-headless_17.0.20%2b8-1%7e24.04_amd64.deb  404  Not Found [IP: 91.189.91.82 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when, count, isnan

spark = SparkSession.builder \
    .appName("DataLakeAssignment") \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 4.0.4


In [3]:
from google.colab import files

uploaded = files.upload()

Saving sales.csv to sales.csv


In [4]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("sales.csv")

df.show(10, truncate=False)
df.printSchema()

print("Total rows:", df.count())
print("Total columns:", len(df.columns))

+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name  |product              |category |quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+---------------+---------------------+---------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|100001  |C0256      |Keerthi Mishra |Saree                |Clothing |2       |6710      |20              |Credit Card     |Mumbai    |Maharashtra   |2026-01-01|Delivered   |
|100002  |C0042      |Arjun Singh    |T-Shirt              |Clothing |6       |680       |5               |Debit Card      |Chennai   |Tamil Nadu    |2025-08-03|Delivered   |
|100003  |C0348      |Keerthi Rao    |Hoodie               |Clothing |6       |3080      |12              |UPI             |P

In [5]:
import os
import shutil

os.makedirs("bronze", exist_ok=True)

shutil.copyfile("sales.csv", "bronze/sales.csv")

print("Raw dataset saved in Bronze layer")

Raw dataset saved in Bronze layer


In [6]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|           10|      0|       0|       0|         0|               0|             8|   8|    0|         0|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [7]:
total_rows = df.count()
unique_rows = df.dropDuplicates().count()

print("Total rows:", total_rows)
print("Unique rows:", unique_rows)
print("Duplicate rows:", total_rows - unique_rows)

Total rows: 1000
Unique rows: 990
Duplicate rows: 10


In [8]:
df.describe().show()
df.show(20, truncate=False)

+-------+------------------+-----------+-------------+---------+--------+------------------+------------------+------------------+----------------+-----------+--------------+----------+------------+
|summary|          order_id|customer_id|customer_name|  product|category|          quantity|        unit_price|  discount_percent|  payment_method|       city|         state|order_date|order_status|
+-------+------------------+-----------+-------------+---------+--------+------------------+------------------+------------------+----------------+-----------+--------------+----------+------------+
|  count|              1000|       1000|          990|     1000|    1000|              1000|              1000|              1000|             992|        992|          1000|      1000|        1000|
|   mean|        100497.072|       NULL|         NULL|     NULL|    NULL|             5.355|            5702.4|            19.948|            NULL|       NULL|          NULL|      NULL|        NULL|
| std

In [9]:
silver_df = df

for c, t in silver_df.dtypes:
    if t == "string":
        silver_df = silver_df.withColumn(
            c,
            when(trim(col(c)) == "", None)
            .otherwise(trim(col(c)))
        )

silver_df = silver_df.dropDuplicates()

silver_df.show(10, truncate=False)

+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category       |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+------------------+----------------------+---------------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|100331  |C0005      |Sakshi Joshi      |Football              |Sports         |2       |670       |30              |Credit Card     |Bengaluru |Karnataka  |2025-06-06|Cancelled   |
|100333  |C0263      |Siddharth Kulkarni|Programming Book      |Books          |4       |670       |0               |Net Banking     |Pune      |Maharashtra|2025-05-08|Delivered   |
|100506  |C0228      |Diya Tripathi     |Competitive Exam Guide|Books          |6       |6

In [ ]:
print("Original row count:", df.count())
print("Silver row count:", silver_df.count())

silver_df.printSchema()

silver_df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
]).show()

In [10]:
import glob

os.makedirs("silver", exist_ok=True)

silver_df.coalesce(1).write \
    .option("header", True) \
    .mode("overwrite") \
    .csv("silver_temp")

part_file = glob.glob("silver_temp/part-*.csv")[0]

shutil.copyfile(part_file, "silver/sales_silver.csv")

print("Silver layer saved successfully")

Silver layer saved successfully


In [11]:
files.download("silver/sales_silver.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>